# Bronze Layer Loading

## Purpose

Load raw source CSV files into Bronze Delta tables while preserving source fidelity.

## inputs

- Landing CSV files
- Profiling metadata

## Outputs

- Bronze Delta tables

## Design Practices

- No data cleaing
- No deduplication
- Preserve raw values
- Add ingestion metadata

# Configuration and Run Metadata

Define source locations, table mappings and execution metadata for the Bronze load.

In [1]:
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType, TimestampType)

StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 3, Finished, Available, Finished, False)

In [2]:
BRONZE_RUN_ID = str(uuid.uuid4())
BRONZE_STARTED_AT = datetime.now(timezone.utc)
BRONZE_START_PERF = time.perf_counter()

LOAD_TYPE = "FULL"

SOURCE_RELATIVE_PATH = (
    "Files/landing/olist/source_csv"
)

SOURCE_LOCAL_PATH = Path(
    "/lakehouse/default/Files/landing/olist/source_csv/"
)

print("Bronze run ID: ", BRONZE_RUN_ID)
print("Started At: ", BRONZE_STARTED_AT)
print("Load Type: ", LOAD_TYPE)

StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 4, Finished, Available, Finished, False)

Bronze run ID:  69f79f3d-a213-4b36-849c-15fcb211bd0b
Started At:  2026-07-28 11:05:21.874241+00:00
Load Type:  FULL


# Source-to-Target Mapping

Map every source CSV file to its corresponding Bronze Delta Table.

In [3]:
SOURCE_TABLE_MAPPING = {
    "olist_customers_dataset.csv": "bronze_customers",
    "olist_geolocation_dataset.csv": "bronze_geolocation",
    "olist_order_items_dataset.csv": "bronze_order_items",
    "olist_order_payments_dataset.csv": "bronze_order_payments",
    "olist_order_reviews_dataset.csv": "bronze_order_reviews",
    "olist_orders_dataset.csv": "bronze_orders",
    "olist_products_dataset.csv": "bronze_products",
    "olist_sellers_dataset.csv": "bronze_sellers",
    "product_category_name_translation.csv": "bronze_category_translation"
}

StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 5, Finished, Available, Finished, False)

In [4]:
actual_source_files = {
    file.name
    for file in SOURCE_LOCAL_PATH.glob("*.csv")
}

expected_source_files = set(SOURCE_TABLE_MAPPING.keys())

missing_files = (
    expected_source_files - actual_source_files
)

unexpected_files = (
    actual_source_files - expected_source_files
)

print("Expected files:", len(expected_source_files))
print("Actual files:", len(actual_source_files))
print("Missing files:", missing_files)
print("Unexpected files:", unexpected_files)

if missing_files: 
    raise FileNotFoundError(
        f"Bronze load cannot continue. "
        f"Missing files: {missing_files}"
    )

print("Source-file validation passed.")

StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 6, Finished, Available, Finished, False)

Expected files: 9
Actual files: 9
Missing files: set()
Unexpected files: set()
Source-file validation passed.


In [7]:
def create_record_hash(dataframe, source_columns):
    normalized_columns = [
        F.coalesce(
            F.col(column_name).cast("string"),
            F.lit("<NULL>")
        )
        for column_name in source_columns
    ]

    return dataframe.withColumn(
        "_bronze_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *normalized_columns
            ),
            256
        )
    )

StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 9, Finished, Available, Finished, False)

# Bronze Load

Read each raw source file as strings, add lineage metadata and write a managed Delta table.

In [9]:
bronze_audit_records = []
bronze_dataframes = {}

for source_file, target_table in (SOURCE_TABLE_MAPPING.items()):
    spark_source_path = (
        f"{SOURCE_RELATIVE_PATH}/"
        f"{source_file}"
    )

    print(
        f"Loading {source_file}"
        f"into {target_table}"
    )

    source_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiline", True)
        .option("quote",'"')
        .option("escape",'"')
        .csv(spark_source_path)
    )

    source_columns = source_df.columns
    source_row_count = source_df.count()
    bronze_df = create_record_hash(source_df, source_columns)

    bronze_df = (
        bronze_df
        .withColumn("_bronze_run_id", F.lit(BRONZE_RUN_ID))
        .withColumn("_bronze_ingested_at", F.current_timestamp())
        .withColumn("_bronze_source_file", F.lit(source_file))
        .withColumn("_bronze_load_type", F.lit(LOAD_TYPE))
    )

    # Place source columns first and metadata last
    bronze_df = bronze_df.select(
        *source_columns,
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at",
        "_bronze_source_file",
        "_bronze_load_type"
    )

    bronze_df.write.mode("overwrite").format("delta").option("overwriteSchema","true").saveAsTable(target_table)

    target_row_count = spark.table(target_table).count()

    load_status = (
        "SUCCESS"
        if source_row_count == target_row_count
        else "FAILED"
    )

    bronze_audit_records.append({
        "bronze_run_id": BRONZE_RUN_ID,
        "source_file": source_file,
        "target_table": target_table,
        "load_type": LOAD_TYPE,
        "source_row_count": int(source_row_count),
        "target_row_count": int(target_row_count),
        "row_count_difference": int(target_row_count - source_row_count),
        "column_count": int(len(source_columns)),
        "load_status": load_status,
        "loaded_at_utc": datetime.now(timezone.utc)
    })

    bronze_dataframes[target_table] = bronze_df
    print(
        f"Completed {target_table}: "
        f"{target_row_count:,} rows"
    )

StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 11, Finished, Available, Finished, False)

Loading olist_customers_dataset.csvinto bronze_customers
Completed bronze_customers: 99,441 rows
Loading olist_geolocation_dataset.csvinto bronze_geolocation
Completed bronze_geolocation: 1,000,163 rows
Loading olist_order_items_dataset.csvinto bronze_order_items
Completed bronze_order_items: 112,650 rows
Loading olist_order_payments_dataset.csvinto bronze_order_payments
Completed bronze_order_payments: 103,886 rows
Loading olist_order_reviews_dataset.csvinto bronze_order_reviews
Completed bronze_order_reviews: 99,224 rows
Loading olist_orders_dataset.csvinto bronze_orders
Completed bronze_orders: 99,441 rows
Loading olist_products_dataset.csvinto bronze_products
Completed bronze_products: 32,951 rows
Loading olist_sellers_dataset.csvinto bronze_sellers
Completed bronze_sellers: 3,095 rows
Loading product_category_name_translation.csvinto bronze_category_translation
Completed bronze_category_translation: 71 rows


In [10]:
failed_loads = [
    record
    for record in bronze_audit_records
    if record["load_status"] != "SUCCESS"
]

if failed_loads:
    raise RuntimeError(
        f"Bronze reconciliation failed: "
        f"{failed_loads}"
    )
print(
    f"All {len(bronze_audit_records)} "
    "Bronze tables loaded successfully."
)

StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 12, Finished, Available, Finished, False)

All 9 Bronze tables loaded successfully.


# Bronze Load Audit

Record source-to-target reconciliation and execution metadata for every Bronze table.

In [11]:
bronze_audit_schema = StructType([
    StructField("bronze_run_id", StringType(), False),
    StructField("source_file", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("source_row_count", LongType(), False),
    StructField("target_row_count", LongType(), False),
    StructField("row_count_difference", LongType(), False),
    StructField("column_count", LongType(), False),
    StructField("load_status", StringType(), False),
    StructField("loaded_at_utc", TimestampType(), False),
])

bronze_audit_df = spark.createDataFrame(bronze_audit_records, schema = bronze_audit_schema)

display(bronze_audit_df.orderBy("target_table"))

StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 592e1c1b-a043-4d41-a392-d1aff56215cb)

In [13]:
bronze_audit_df.write.mode("append").format("delta").saveAsTable("audit_bronze_load")

StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 15, Finished, Available, Finished, False)

In [15]:
import builtins

BRONZE_COMPLETED_AT = datetime.now(timezone.utc)

BRONZE_DURATION_SECONDS = builtins.round(
    time.perf_counter() - BRONZE_START_PERF, 2
)

bronze_run_history = [{
    "bronze_run_id": BRONZE_RUN_ID,
    "started_at_utc": BRONZE_STARTED_AT,
    "completed_at_utc": BRONZE_COMPLETED_AT,
    "duration_seconds": BRONZE_DURATION_SECONDS,
    "load_type": LOAD_TYPE,
    "source_file_count": len(SOURCE_TABLE_MAPPING),
    "target_table_count": len(bronze_audit_records),
    "successful_table_count": sum(
        record["load_status"] == "SUCCESS"
        for record in bronze_audit_records
    ),
    "failed_table_count": len(failed_loads),
    "status": (
        "SUCCESS"
        if not failed_loads
        else "FAILED"
    )
}]

bronze_run_history_df = spark.createDataFrame(bronze_run_history)

display(bronze_run_history_df)

bronze_run_history_df.write.mode("append").format("delta").saveAsTable("audit_bronze_run_history")


StatementMeta(, 3087b4ea-868f-49f4-bdd2-b523b4696fb4, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6fb63a8d-384d-4ff1-98be-de2de39e19c1)